# 04 — Economic damage analysis (FIAT)

**Goal**: Load FIAT building-level damage output, explore the spatial distribution of losses, and compute aggregate statistics. Then compare damage across two scenarios (factual vs. a climate counterfactual).

**Prerequisite**: A completed FIAT run (the FIAT binary must have been executed for each scenario you want to compare). The result file is `output/spatial.fgb`.  
**No simulation is run here** — this notebook only reads existing outputs.

---

## What FIAT produces

FIAT (Flood Impact Assessment Tool) computes building-level economic damage by applying depth-damage functions to the flood hazard map. Its main output is:

| File | Description |
|------|-------------|
| `output/spatial.fgb` | **FlatGeobuf** — one polygon per building, with damage columns and geometry. The primary output. |
| `output/output.csv` | Tabular summary (same columns, no geometry). |
| `exposure/exposure.csv` | Building exposure table: object_id, max_damage_total, land use, etc. |
| `output/output_relative_damage.fgb` | Like spatial.fgb but with an additional `relative_damage` column (0–1). **Requires** running `fiat_add_relative_damage.py` first. |
| `output/spatial_with_pop_and_flood.fgb` | Like spatial.fgb but with allocated WorldPop population. **Requires** running `fiat_add_pop_exposed_metric.py` first. |

### Key columns in `spatial.fgb`

| Column | Description |
|--------|-------------|
| `object_id` | Unique building identifier |
| `total_damage` | Absolute economic damage [USD] |
| `_damage` | Damage fraction applied (0–1) |
| `_depth` | Inundation depth at the building [m] |
| `geometry` | Building polygon (or centroid, depending on FIAT version) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import contextily as ctx
from pathlib import Path

In [ ]:
# ── Edit these for your event ────────────────────────────────────────────────
BASE   = Path("/p/11210471-001-compass/03_Runs")
REGION = "sofala"
EVENT  = "Idai"

FACTUAL_FOLDER = "event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0"
CF_RAIN_FOLDER = "event_tp_era5_hourly_zarr_CF-8_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0"
# ─────────────────────────────────────────────────────────────────────────────

FIAT_ROOT = BASE / REGION / EVENT / "fiat"

FACTUAL_FGB = FIAT_ROOT / FACTUAL_FOLDER / "output" / "spatial.fgb"
CF_RAIN_FGB = FIAT_ROOT / CF_RAIN_FOLDER / "output" / "spatial.fgb"

for label, p in [("Factual", FACTUAL_FGB), ("CF rain −8%", CF_RAIN_FGB)]:
    status = "✓" if p.exists() else "⚠️  NOT FOUND"
    print(f"{status}  {label}: {p}")

## Loading the damage GeoDataFrame

`geopandas.read_file` reads FlatGeobuf (`.fgb`) directly — no conversion needed. We start with the factual run.

In [ ]:
gdf = gpd.read_file(FACTUAL_FGB)

print(f"CRS          : {gdf.crs}")
print(f"Total records: {len(gdf):,}  buildings")
print(f"Columns      : {list(gdf.columns)}")
print()
gdf.head(3)

## Exploring the damage distribution

Not every building in the model domain is flooded. We first filter to affected buildings (`total_damage > 0`) and inspect the distribution.

In [ ]:
gdf_dmg = gdf[gdf["total_damage"] > 0].copy()

n_total   = len(gdf)
n_damaged = len(gdf_dmg)
print(f"Affected buildings : {n_damaged:,}  ({100*n_damaged/n_total:.1f}% of all buildings in domain)")
print(f"Total damage       : USD {gdf_dmg['total_damage'].sum():,.0f}")
print(f"Mean damage/bldg   : USD {gdf_dmg['total_damage'].mean():,.0f}")
print(f"Median damage/bldg : USD {gdf_dmg['total_damage'].median():,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

damage_vals = gdf_dmg["total_damage"].values
ax.hist(damage_vals, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xscale("log")
ax.set_xlabel("Damage per building [USD]  (log scale)", fontsize=10)
ax.set_ylabel("Number of buildings", fontsize=10)
ax.set_title(f"TC {EVENT} — distribution of building damage (factual)", fontsize=11)

# Annotate mean and median
ax.axvline(damage_vals.mean(),   color="red",    linestyle="--", linewidth=1.2, label=f"Mean   USD {damage_vals.mean():,.0f}")
ax.axvline(np.median(damage_vals), color="orange", linestyle="--", linewidth=1.2, label=f"Median USD {np.median(damage_vals):,.0f}")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## Mapping building damage spatially

We plot each building as a dot coloured by `total_damage`. Use centroids so the scatter doesn't overlap too much in dense urban areas.

In [ ]:
# Use centroids for plotting (works even if geometry is already a point)
gdf_dmg_plot = gdf_dmg.copy()
if gdf_dmg_plot.geometry.geom_type.unique()[0] != "Point":
    gdf_dmg_plot["geometry"] = gdf_dmg_plot.geometry.centroid

# Reproject to Web Mercator for contextily
gdf_web = gdf_dmg_plot.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(9, 8))

sc = gdf_web.plot(
    column="total_damage",
    ax=ax,
    markersize=3,
    cmap="YlOrRd",
    norm=mcolors.LogNorm(
        vmin=max(1, gdf_web["total_damage"].min()),
        vmax=gdf_web["total_damage"].max(),
    ),
    legend=True,
    legend_kwds={"label": "Building damage [USD]", "shrink": 0.6},
    alpha=0.8,
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=11)
ax.set_title(f"TC {EVENT} — spatial distribution of building damage (factual)", fontsize=11)
ax.set_xlabel("Easting [m]")
ax.set_ylabel("Northing [m]")
plt.tight_layout()

## Relative damage (if available)

`relative_damage` expresses damage as a fraction of the building's **replacement value** (0 = no damage, 1 = total destruction). It requires running `fiat_add_relative_damage.py` first, which merges `spatial.fgb` with the `exposure.csv` max-damage values.

If the column is present, we plot its distribution.

In [ ]:
# Try loading the enriched file; fall back to spatial.fgb if not available
rel_dmg_path = FACTUAL_FGB.parent / "output_relative_damage.fgb"

if rel_dmg_path.exists():
    gdf_rel = gpd.read_file(rel_dmg_path)
    gdf_rel_dmg = gdf_rel[gdf_rel["relative_damage"] > 0]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(gdf_rel_dmg["relative_damage"], bins=40, color="#E67E22", edgecolor="white")
    ax.set_xlabel("Relative damage (fraction of replacement value)", fontsize=10)
    ax.set_ylabel("Number of buildings", fontsize=10)
    ax.set_title(f"TC {EVENT} — relative damage distribution (factual)", fontsize=11)
    ax.axvline(gdf_rel_dmg["relative_damage"].mean(), color="red", linestyle="--",
               label=f"Mean = {gdf_rel_dmg['relative_damage'].mean():.2f}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
else:
    print("relative_damage column not found.")
    print(f"Run  postprocessing/fiat/fiat_add_relative_damage.py  to generate {rel_dmg_path}")

## Comparing damage across scenarios

We repeat the same analysis for the counterfactual run (−8% precipitation) and compare total damage, number of affected buildings, and mean damage per building.

In [ ]:
def damage_metrics(fgb_path: Path) -> dict:
    """Compute aggregate damage metrics from a FIAT spatial.fgb output."""
    gdf_all = gpd.read_file(fgb_path)
    gdf_hit = gdf_all[gdf_all["total_damage"] > 0]
    return {
        "total_damage_usd": gdf_hit["total_damage"].sum(),
        "n_affected":        len(gdf_hit),
        "mean_damage_usd":   gdf_hit["total_damage"].mean() if len(gdf_hit) > 0 else 0.0,
    }

scenarios = {
    "Factual":   FACTUAL_FGB,
    "CF rain −8%": CF_RAIN_FGB,
}

all_metrics = {}
for label, path in scenarios.items():
    if path.exists():
        all_metrics[label] = damage_metrics(path)
    else:
        print(f"⚠️  Skipping {label}: {path} not found")

# Summary table
print(f"{'Scenario':20s} | {'Total damage [M USD]':>20} | {'Affected bldgs':>14} | {'Mean damage [USD]':>17}")
print("-" * 80)
for label, m in all_metrics.items():
    print(f"{label:20s} | {m['total_damage_usd']/1e6:>20.2f} | {m['n_affected']:>14,} | {m['mean_damage_usd']:>17,.0f}")

In [ ]:
if len(all_metrics) < 2:
    print("Need at least two scenarios to compare — check that both paths exist.")
else:
    metric_keys   = ["total_damage_usd", "n_affected", "mean_damage_usd"]
    metric_labels = ["Total damage [M USD]", "Affected buildings", "Mean damage/building [USD]"]
    scale_factors = [1e6, 1, 1]  # for display

    labels  = list(all_metrics.keys())
    colors  = ["#2196F3", "#FF9800"]
    x       = np.arange(len(metric_keys))
    width   = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))

    for i, (label, col) in enumerate(zip(labels, colors)):
        vals = [all_metrics[label][k] / sf for k, sf in zip(metric_keys, scale_factors)]
        bars = ax.bar(x + i * width, vals, width=width, label=label, color=col, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.02,
                    f"{v:,.1f}" if v < 1000 else f"{v:,.0f}",
                    ha="center", va="bottom", fontsize=8)

    # Attribution label between the first pair of bars
    if "total_damage_usd" in metric_keys:
        fact_val = all_metrics[labels[0]]["total_damage_usd"]
        cf_val   = all_metrics[labels[1]]["total_damage_usd"]
        pct      = (fact_val - cf_val) / fact_val * 100
        ax.annotate(
            f"+{pct:.1f}% from climate change",
            xy=(0 + width / 2, fact_val / 1e6 * 1.05),
            fontsize=9, color="red", ha="center"
        )

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(metric_labels, fontsize=10)
    ax.set_title(f"TC {EVENT} — economic damage attribution", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

---

## Summary

You have now:
1. Loaded FIAT building-level output (`spatial.fgb`)
2. Explored the damage distribution and mapped it spatially
3. Compared total, mean, and count-of-affected-buildings across a factual and counterfactual scenario

## Next steps

- **Population exposure**: run `fiat_add_pop_exposed_metric.py` to add WorldPop population to each building, then explore `spatial_with_pop_and_flood.fgb` (see `postprocessing/attribution/population_attribution.py`).
- **Relative damage**: run `fiat_add_relative_damage.py` to add `relative_damage` (fraction of max value) for cross-region comparisons.
- **Full event set**: the scripts in `postprocessing/attribution/` (`damage_attribution.py`) automate multi-event comparison and save summary CSVs to `/p/11210471-001-compass/04_Results/`.